# Welcome to Week 4 - LangChain and LangGraph

## Lab 1: Abstraction Levels and the building blocks

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The LangChain docs</h2>
            <span style="color:#00bfff;">The documentation lives at <a href="https://docs.langchain.com/oss/python/">https://docs.langchain.com/oss/python/</a>. It is worth a browse. Watch out for older 0.x material that still floats around the web, because the modern API is quite different.
            </span>
        </td>
    </tr>
</table>

## The four levels of abstraction

LangChain and LangGraph form 4 levels of abstraction, with each one built on the ones before. The terminology is confusing because 'LangChain' appears in a few places..

| Layer | Packages | What it gives you | What you control |
|---|---|---|---|
| 1. Building blocks | `langchain-core` + `langchain-openai` | chat models, the `@tool` decorator, messages, structured output | everything, including the tool loop by hand |
| 2. Orchestration | `langgraph` | a graph of steps, with state, memory and checkpointing | the control flow (you design the graph) |
| 3. Agent | `langchain` (`create_agent`) | the standard agent loop, prebuilt | just model, tools and a prompt |
| 4. Harness | `deepagents` (`create_deep_agent`) | an opinionated harness with planning, sub-agents and a filesystem | your intent |


## Order of play for this week:

DAY 1 (today): The building blocks  
DAY 2: LangGraph  
DAY 3: LangChain create_agent  
DAY 4: Deep Agents  
DAY 5: The Sidekick project  

## Today is Layer 1: the building blocks

This is where LangChain began: an abstraction layer, not unlike LiteLLM - but a more heavyweight version.

In [3]:
#simple example for myself
from langchain_openai import ChatOpenAI
model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

messages = [
    (
        'system',
        'You are a helpful translator, translate the user sentence to Spanish'
    ),
    ('human', 'Where is the bathroom?')
]


response = model.invoke(messages)
print(response.content)



¿Dónde está el baño?


In [17]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
# say I want to use langchain_openai package my team would tell me, note here we are using langchain_core package so to use it I am searching for that package on good, I have done this with node work you just import the package and needed function and can start to use them
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True)

True

### A first model call

`ChatOpenAI` is the abstraction around OpenAI calls, and there are similar packages for other LLM providers.

We call `invoke` with a prompt; this is a key method in LangChain.

In [15]:
# llm = ChatOpenAI(model="gpt-5.4-mini")
llm = ChatOpenAI(model="gpt-5.4-mini")
# message = "In 1 sentence, what does it mean for an AI Agent to be autonomous"
message = 'In 1 sentence, what does it mean for an AI Agent to be autonomous'
# reply = llm.invoke(message)
reply = llm.invoke(message)

# print(reply.content)
print(reply.content)




An AI agent is autonomous if it can independently perceive its environment, make decisions, and take actions toward a goal with little or no human intervention.


### Streaming

For a live, token by token feel, swap `invoke` for `stream` and loop over the chunks

In [9]:
# for chunk in llm.stream("Tell me a two line poem about autonomous agents"):
#     print(chunk.content, end="", flush=True)
# for live token by token feel swap invoke for stream and loop over the chunks
for chunk in llm.stream("Tell me a 2 line poem about autonomous agents"):
    print(chunk.content, end="", flush=True)

Autonomous agents roam where goals are set,  
Learning in silence, they adapt and never forget.

### Any OpenAI-compatible provider

As before; we can use OpenAI compatible endpoints with the ChatOpenAI object

In [19]:
# openrouter_llm = ChatOpenAI(
#     model="anthropic/claude-haiku-4.5",
#     base_url="https://openrouter.ai/api/v1",
#     api_key=os.getenv("OPENROUTER_API_KEY"),
# )
openrouter_llm = ChatOpenAI(
    model="anthropic/claude-haiku-4.5",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    max_tokens=1000
)

# reply = openrouter_llm.invoke("In one sentence, what is LangChain?")
# print(reply.content)
reply = openrouter_llm.invoke("in one sentence, what is LangChain?")
print(reply.content)

LangChain is a framework for developing applications powered by large language models (LLMs) through composable modules and chains that simplify building complex AI workflows.


### Messages

LangChain comes with abstractions around SystemMessage, HumanMessage, AIMessage, although you can use the usual list-of-dicts instead.

In [20]:
# messages = [
#     SystemMessage("You are a terse assistant who answers in exactly five words."),
#     HumanMessage("What is the capital of France?"),
# ]
messages = [
    SystemMessage("You are a terse assistant who answers in exactly 5 words."),
    HumanMessage("What is the capital of France?"),
]

# print(llm.invoke(messages).content)
print(llm.invoke(messages).content)

# The exact same call using plain dictionaries, the format you already know
messages_as_dicts = [
    {"role": "system", "content": "You are a terse assistant who answers in exactly five words."},
    {"role": "user", "content": "What is the capital of France?"},
]
print(llm.invoke(messages_as_dicts).content)

Paris is France’s capital city.
Paris is France's capital city.


### Tools with the `@tool` decorator

A tool is a Python function the model is allowed to call. The modern way to make one is the `@tool` decorator. Your docstring becomes the description the model reads, and your type hints become the argument schema, just like `@function_tool` with OpenAI Agents SDK.

This replaces the older `Tool(...)` wrapper from earlier versions of LangChain.

In [43]:
@tool
def get_share_price(symbol: str) -> float:
    """Return the current share price for a given ticker symbol."""
    fake_prices = {"AAPL": 241.5, "GOOG": 168.2, "AMZN": 198.0}
    aliases = {"GOOGL": "GOOG"}
    ticker =  aliases.get(symbol.upper(), symbol.upper())
    return fake_prices.get(ticker.upper(), 0.0)

print("name:", get_share_price.name)
print("description:", get_share_price.description)
print("args:", get_share_price.args)
print("called directly:", get_share_price.invoke({"symbol": "AAPL"}))

name: get_share_price
description: Return the current share price for a given ticker symbol.
args: {'symbol': {'title': 'Symbol', 'type': 'string'}}
called directly: 241.5


### Giving tools to the model

This is a bit clunky; redemption will come in Day 3 when we use the create_agent features,

For now, we have to write the loop ourselves, rather like we did in Week 1.

The first step is to bind the tools to the model with `bind_tools`. Now when we invoke, the model may come back not with an answer but with a request to run a tool. That request shows up in `.tool_calls`.

In [23]:
# llm_with_tools = llm.bind_tools([get_share_price])
llm_with_tools = llm.bind_tools([get_share_price])

# response = llm_with_tools.invoke("What is the share price of Amazon?")
response = llm_with_tools.invoke("what is the share price of Amazon?")
# print("content:", repr(response.content))
print("content:", repr(response.content))
print('response;', response)
print("tool_calls:", response.tool_calls)

content: ''
response; content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 138, 'total_tokens': 157, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EF67uaDF63OVAj0XmQf2prByCWc8v', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a0216c-d66b-7752-95e9-f2a5cfe717e6-0' tool_calls=[{'name': 'get_share_price', 'args': {'symbol': 'AMZN'}, 'id': 'call_9V457dTC0wYNMCELuIf4hvgi', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 138, 'output_tokens': 19, 'total_tokens': 157, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
tool_calls:

### Running the tool loop by hand

So now we need to write a little loop, quite similar to Week 1

In [31]:
# Start the conversation and keep the model's tool request in the history
conversation = [HumanMessage("What is the share price of Amazon?")]
ai_message = llm_with_tools.invoke(conversation)
conversation.append(ai_message)
print('conversation;', conversation)

# Run each requested tool and add its result as a ToolMessage
for call in ai_message.tool_calls:
    if call["name"] == "get_share_price":
        result = get_share_price.invoke(call["args"])
        conversation.append(ToolMessage(content=str(result), tool_call_id=call['id']))
    

# Invoke again, now that the model can see the tool result
# final = llm_with_tools.invoke(conversation)
print("Conversation:2", conversation)
final = llm_with_tools.invoke(conversation)
print(final.content)

conversation; [HumanMessage(content='What is the share price of Amazon?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 138, 'total_tokens': 157, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EF6Xt2GqHDHbowPtIE5dqi09PhJEa', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a02185-70ba-7e33-9872-1fdb9149c1cc-0', tool_calls=[{'name': 'get_share_price', 'args': {'symbol': 'AMZN'}, 'id': 'call_osMKJbMgN7KeqYNCqpArdZqd', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 138, 'output_tokens': 19, 'total_tokens': 157, 'input

### Structured output

Similar to OpenAI Agents SDK, we can require the model to respond with a Pydantic subclass

In [33]:
# class Company(BaseModel):
#     name: str = Field(description="The company name")
#     ticker: str = Field(description="The stock ticker symbol")
#     founded_year: int = Field(description="The year the company was founded")

#output
class Company(BaseModel):
    name: str = Field(description="The company name")
    ticker: str = Field(description="The stock ticker symbol")
    founded_year: int = Field(description="The year the company was founded")

# structured_llm = llm.with_structured_output(Company)
structured_llm = llm.with_structured_output(Company)

# company = structured_llm.invoke("Tell me about Amazon the technology company")
company = structured_llm.invoke("Tell me about Amazon the technology company")
print(company)
print("Just the ticker:", company.ticker)

name='Amazon' ticker='AMZN' founded_year=1994
Just the ticker: AMZN


## That is Layer 1

It's like a more rich and more involved version of LiteLLM.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Write a second tool of your own, perhaps one that looks up a fictional weather report for a city. Bind both tools to the model and ask a question that needs both, then run the tool loop by hand until the model gives a final answer. It will feel more gritty than OpenAI Agents SDK or CrewAI.
            We will fix that by Day 3!
            </span>
        </td>
    </tr>
</table>

In [47]:
@tool
def get_the_temp(location: str) -> int:
    """Get the weather for a location"""
    weather = {"Portland": 90, "Seattle": 70, "SF": 60}
    return weather.get(location, 0)

llm_with_tools = llm.bind_tools([get_share_price, get_the_temp])
messages = [HumanMessage("What is the price of Google? what is the temp in Portland?")]

ai_response = llm_with_tools.invoke(messages)
messages.append(ai_response)
print(len(ai_response.tool_calls), response)

for call in ai_response.tool_calls:
    print(call)
    if call["name"] == "get_share_price":
        result = get_share_price.invoke(call['args'])
        messages.append(ToolMessage(content=str(result), tool_call_id=call['id']))
    if call['name'] == 'get_the_temp':
        result = get_the_temp.invoke(call['args'])
        messages.append(ToolMessage(content=str(result), tool_call_id=call['id']))

full_response = llm_with_tools.invoke(messages)
print("---")
print(full_response.content)

    


2 content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 166, 'total_tokens': 220, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EF8h7eegjV6a3SggVXPOmviYckaSZ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a02203-7385-74c3-8601-c97b0d6ba980-0' tool_calls=[{'name': 'get_share_price', 'args': {'symbol': 'GOOGL'}, 'id': 'call_cVmYJi5ZcayAbT7SDPaXc8IU', 'type': 'tool_call'}, {'name': 'get_the_temp', 'args': {'location': 'San Francisco, CA'}, 'id': 'call_sJo5wj7HjS39ErLQRxUGqOyV', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 166, 'output_tokens': 54, 'total_tokens': 220, 'input_to